# Aprendizado por Reforço

---

**Professor:** Prof. Gabriel Lima  
**Aula:** 02  
**Exercício:** 2A  

---

### Objetivo :  
Modelar um MDP a partir de métodos computacionais.

## Bibliotecas Utilizadas

Utilizaremos bibliotecas triviais para a construção do MDP.

In [ ]:
import numpy as np
from numpy.linalg import inv

## Definição do MRP

Para este exercício, vamos considerar o seguinte MRP:

<img src="mrp_aula.png" alt="IMG" width="800"/>


Podemos interpretá-lo como sendo o timeframe de um estudante que pode transitar entre os estados descritos.

Percebam que nesse diagrama não há ações, uma vez que apenas os estados e as recompensas estão representadas.

Dessa forma podemos criar um objeto que encapsula todas essas informações:

In [ ]:
class MRP:

    def __init__(self, StateSpace:list, TransitionMatrix:np.array,Rewards:list,gamma:float):
        self.StateSpace = StateSpace
        self.TransitionMatrix = TransitionMatrix
        self.rewards  = Rewards
        self.gamma = gamma

    
    def Vf(self):
        n_states = len(self.StateSpace)
        I = np.identity(n_states)
        V = np.dot(inv(I-(self.gamma*self.TransitionMatrix)),self.rewards)
        return dict(zip(self.StateSpace,V))


Sabemos que : 

$$
v = \mathcal{R} + \gamma \mathcal{P} v
$$


Portanto, se resolvermos analiticamente, temos:  

$$
(I - \gamma \mathcal{P}) v = \mathcal{R} \quad \Rightarrow \quad v = (I - \gamma \mathcal{P})^{-1} \mathcal{R}
$$

onde :  

$$
u \cdot v = \sum_{i=1}^{n} u_i v_i
$$

e,  


$$
I_n = \begin{bmatrix}
1 & 0 & \cdots & 0 \\
0 & 1 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & 1
\end{bmatrix}
$$

**A equação acima é computacionalmente custosa !!**

## Definição da matriz de Transição

Analisando o diagrama, sabemos que para esse problema a matriz de transição de estados é dada por: 

$$
\mathcal{P} =
\begin{array}{c|ccccccc}
       & \text{Tiktok} & \text{C1} & \text{C2} & \text{C3} & \text{Party} & \text{Home} & \text{Sleep} \\
\hline
\text{Tiktok} & 0.9 & 0.1 & 0 & 0 & 0 & 0 & 0 \\
\text{C1}     & 0.5 & 0 & 0.5 & 0 & 0 & 0 & 0 \\
\text{C2}     & 0 & 0 & 0 & 0.8 & 0 & 0 & 0.2 \\
\text{C3}     & 0 & 0 & 0 & 0 & 0.4 & 0.6 & 0 \\
\text{Party}  & 0 & 0.2 & 0.4 & 0.4 & 0 & 0 & 0 \\
\text{Home}   & 0 & 0 & 0 & 0 & 0 & 0 & 1 \\
\text{Sleep}  & 0 & 0 & 0 & 0 & 0 & 0 & 1
\end{array}
$$


In [ ]:
TransitionMatrix = np.array([[0.9, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0],
                             [0.5, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0],
                             [0.0, 0.0, 0.0, 0.8, 0.0, 0.0, 0.2],
                             [0.0, 0.0, 0.0, 0.0, 0.4, 0.6, 0.0],
                             [0.0, 0.2, 0.4, 0.4, 0.0, 0.0, 0.0],
                             [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
                             [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]])

Com as seguintes recompensas: 

In [ ]:
StateSpace = ["Tiktok","c1","c2","c3","party","home","sleep"]
Rewards    = [-1,-2,-2,-2,1,10,0]
gamma      = 0.9

## Solução do MRP

Logo, podemos resolver o MRP instanciando a classe previamente criada.

In [ ]:
studentMRP = MRP(StateSpace,TransitionMatrix,Rewards,gamma)

In [ ]:
studentMRP.Vf()

Obtendo assim, a função valor de cada estado.

## Definição do MDP

Utilizando agora as três ações possíveis, podemos montar o MDP:

<img src="mdp_aula.png" alt="IMG" width="800"/>

In [ ]:
ActionSpace = ["Procrastination","Study","Sleeping"]

Onde, pode-se observar que há diferentes possibilidades para diferentes estados.

In [ ]:
class MDP:

    def __init__(self,StateSpace:list,ActionSpace:list,TransitionMatrix:list,Rewards:np.array,gamma:float):
        self.StateSpace = StateSpace
        self.ActionSpace = ActionSpace
        self.TransitionMatrix = TransitionMatrix
        self.Rewards = Rewards 
        self.gamma   = gamma

    def Vf(self,policy):
        n = len(self.StateSpace)
        m = len(self.ActionSpace)

        MRP_Transition_matrix  = np.zeros((n,n))
        MRP_Rewards            = np.zeros(n)

        # Build MRP Transition Matrix
        #current state /line               
        for state in range(0,n):
            #next state /column
            for next_state in range(0,n):
                prob = 0
                for idx in range(0,m):
                    
                    if (policy[state][idx] is not None):
                        actual_transition_Matrix = self.TransitionMatrix[idx]
                        prob += policy[state][idx]*actual_transition_Matrix[state][next_state]
                
                MRP_Transition_matrix[state][next_state] = prob

        for state in range(0,n):
            reward = 0
            for idx in range(0,m):
                if (policy[state][idx] is not None):
                    r = self.Rewards[state][idx]
                    reward += policy[state][idx]*r
            
            MRP_Rewards[state] = reward

        
        mrp = MRP(StateSpace,MRP_Transition_matrix,MRP_Rewards,self.gamma)

        return mrp.Vf() 

In [ ]:
# MDP Transition Probabilities

# a1 = Procrastination
P_a1 = np.array([[   1,   0,    0,    0,    0,    0,    0],
                 [   1,   0,    0,    0,    0,    0,    0],
                 [   0,   0,    0,    0,    0,    0,    1],
                 [   0,   0,    0,    0,    1,    0,    0],
                 [None, None, None, None, None, None, None],
                 [None, None, None, None, None, None, None],
                 [None, None, None, None, None, None, None]])


# a2 = Study
P_a2 = np.array([[   0,   1,    0,    0,    0,    0,    0],
                 [   0,   0,    1,    0,    0,    0,    0],
                 [   0,   0,    0,    1,    0,    0,    0],
                 [None, None, None, None, None, None, None],
                 [   0,  0.2,  0.4,  0.4,    0,    0,    0],
                 [None, None, None, None, None, None, None],
                 [None, None, None, None, None, None, None]])


# a3 = Sleeping
P_a3 = np.array([[None, None, None, None, None, None, None],
                 [None, None, None, None, None, None, None],
                 [None, None, None, None, None, None, None],
                 [   0,   0,    0,    0,    0,    1,     0],
                 [None, None, None, None, None, None, None],
                 [   0,   0,    0,    0,    0,    0 ,    1],
                 [   0,   0,    0,    0,    0,    0 ,    1]])
 

Matrices = [P_a1,P_a2,P_a3]

In [ ]:
# Random Policy 

# Random Policy
Pi_random = np.array([[0.5,  0.5,  None],
                      [0.5,  0.5,  None],
                      [0.5,  0.5,  None],
                      [0.5, None,   0.5],
                      [None,   1,  None],
                      [None,None,     1],
                      [None,None,     1]])

In [ ]:
# Rewards 

R         = np.array([[-1,  -2,  None],
                      [-1,  -2,  None],
                      [ 0,  -2,  None],
                      [ 1, None,   0],
                      [None,  -2,None],
                      [None,None,   10],
                      [None,None,   0]])

In [ ]:
mdp = MDP(StateSpace, ActionSpace, Matrices, R, gamma=0.9)

In [ ]:
mdp.Vf(Pi_random)